# Day 2 — NumPy, Pandas, Matplotlib & SciPy
**30-Day ML & AI Mastery Plan · Shubham**

| Part | What you'll learn | Time |
|---|---|---|
| 1 | **NumPy**: arrays, vectorisation, broadcasting, boolean masks | 30 min |
| 2 | **Pandas**: Series, DataFrame, selecting, grouping, joining (Python's SQL) | 35 min |
| 3 | **Day 1 rebuilt**: the marks analyzer in ~10 lines of Pandas | 10 min |
| 4 | **Matplotlib & Seaborn**: the 5 plots you'll use 90% of the time | 20 min |
| 5 | **SciPy**: statistics and optimisation in one line | 10 min |
| **6** | **Live demo: full Titanic EDA** | 45 min |
| 7 | Exercises | 30–60 min |

Run cells top to bottom with **Shift + Enter**. Everything here is pre-installed in Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize

pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 — NumPy
## 1.1 Why NumPy exists: speed
A Python list holds pointers to separate objects scattered in memory. A NumPy array is **one contiguous block of same-typed numbers** (like a C# `double[]`), and its operations run in compiled C. Every ML library (Pandas, scikit-learn, PyTorch) is built on this idea.

In [ ]:
import time

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

t = time.perf_counter(); squares_list = [x * x for x in py_list]; t_list = time.perf_counter() - t
t = time.perf_counter(); squares_arr = np_arr * np_arr;          t_np = time.perf_counter() - t

print(f"list comprehension: {t_list*1000:7.1f} ms")
print(f"numpy vectorised:   {t_np*1000:7.1f} ms   -> {t_list/t_np:.0f}x faster")

## 1.2 Creating arrays: shape, dtype, ndim

In [ ]:
a = np.array([1, 2, 3, 4])                  # 1-D: a vector
M = np.array([[1, 2, 3], [4, 5, 6]])        # 2-D: a matrix (rows x cols)
print(a, a.shape, a.dtype, a.ndim)
print(M, M.shape, M.ndim, sep="\n")

print(np.zeros((2, 3)))                      # all zeros
print(np.ones(3), np.arange(0, 10, 2), np.linspace(0, 1, 5))
print(np.arange(12).reshape(3, 4))           # reshape: same data, new shape
print(np.eye(3))                             # identity matrix (Day 5)

## 1.3 Indexing, slicing and boolean masks
`M[row, col]`, with `:` meaning "all". **Boolean masks** are the most important trick in data work: a condition creates an array of True/False that selects matching elements.

In [ ]:
M = np.arange(1, 13).reshape(3, 4)
print(M)
print("row 0:", M[0])                 # first row
print("col 1:", M[:, 1])              # second column
print("block:\n", M[0:2, 1:3])        # rows 0-1, cols 1-2

marks = np.array([91, 67, 84, 45, 99, 73, 88])
mask = marks >= 75
print(mask)                            # [ True False  True ...]
print(marks[mask])                     # only the passing marks
print(marks[(marks > 60) & (marks < 90)])   # combine with & and |, always with brackets
print((marks >= 75).mean())           # fraction passing: True counts as 1

## 1.4 Vectorisation and broadcasting
Operations apply element-wise with **no loops**. *Broadcasting* lets arrays of different shapes combine: a single number or a single row is stretched to match.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
print(x * 10, x + x, x ** 2, np.sqrt(x), np.exp(x).round(2))

# Broadcasting: (3 students x 4 subjects) - (4 subject means)
scores = np.array([[91, 84, 88, 79],
                   [67, 72, 80, 81],
                   [45, 58, 52, 70]])
subject_means = scores.mean(axis=0)          # axis=0: down the rows -> one value per column
print("subject means:", subject_means)
print("difference from subject mean:\n", scores - subject_means)   # shape (3,4) - (4,) works

## 1.5 Aggregations and the `axis` argument
`axis=0` collapses **rows** (one result per column). `axis=1` collapses **columns** (one result per row). It is the #1 source of confusion, so check the output shape every time.

In [ ]:
print("student averages (axis=1):", scores.mean(axis=1).round(2))
print("subject maxima   (axis=0):", scores.max(axis=0))
print("overall:", scores.mean().round(2), "| best student index:", scores.mean(axis=1).argmax())

## 1.6 Standardisation in one line: a preview of Day 4
Most ML models want each feature centred at 0 with spread 1: **z = (x − mean) / std**. With broadcasting, it is one line for every column at once.

In [ ]:
z = (scores - scores.mean(axis=0)) / scores.std(axis=0)
print(z.round(2))
print("column means ~0:", z.mean(axis=0).round(6), "| column stds:", z.std(axis=0))

In [ ]:
# Random numbers: reproducible with a seed (vital for ML experiments)
rng = np.random.default_rng(seed=42)
print(rng.integers(0, 100, size=5))
print(rng.normal(loc=70, scale=10, size=5).round(1))   # marks ~ N(70, 10)

---
# Part 2 — Pandas
A **DataFrame** is a table: NumPy columns plus labels. A **Series** is one labelled column. As a C# developer, think of it as **an in-memory SQL table with LINQ**.

| SQL / LINQ | Pandas |
|---|---|
| `SELECT name, math` | `df[["name", "math"]]` |
| `WHERE math > 80` / `.Where()` | `df[df.math > 80]` or `df.query("math > 80")` |
| `ORDER BY avg DESC` | `df.sort_values("avg", ascending=False)` |
| `GROUP BY dept` + `AVG()` | `df.groupby("dept")["salary"].mean()` |
| `JOIN ... ON id` | `df.merge(other, on="id", how="left")` |
| `COUNT(*)` per value | `df["col"].value_counts()` |
| `ISNULL` | `df.isna()` |

In [ ]:
df = pd.DataFrame({
    "name":  ["Asha", "Ravi", "Meena", "John", "Priya"],
    "dept":  ["CS", "EE", "CS", "ME", "EE"],
    "math":  [91, 67, 84, 45, 99],
    "physics": [84, 72, 90, 58, 95],
})
df

In [ ]:
print(df.shape, "\n")
print(df.dtypes, "\n")
df.describe()            # count, mean, std, min, quartiles, max for every numeric column

## 2.1 Selecting: columns, rows, `loc` and `iloc`
`loc` selects by **label**; `iloc` selects by **integer position**. Use `loc` with a boolean mask for SQL-style WHERE.

In [ ]:
print(df["math"].head(3))                         # one column -> Series
print(df[["name", "math"]].head(3))               # several columns -> DataFrame
print(df.loc[df["math"] > 80, ["name", "math"]])  # WHERE math > 80 SELECT name, math
print(df.iloc[0:2, 0:2])                          # first 2 rows, first 2 columns
print(df.query("dept == 'CS' and physics >= 85"))

## 2.2 New columns, sorting, groupby, merge

In [ ]:
df["avg"] = df[["math", "physics"]].mean(axis=1)         # vectorised: no loop
df["result"] = np.where(df["avg"] >= 60, "PASS", "FAIL")  # vectorised if/else
print(df.sort_values("avg", ascending=False))

print("\n", df.groupby("dept")["avg"].agg(["mean", "max", "count"]))

depts = pd.DataFrame({"dept": ["CS", "EE", "ME"], "hod": ["Dr. Rao", "Dr. Iyer", "Dr. Khan"]})
print("\n", df.merge(depts, on="dept", how="left")[["name", "dept", "hod"]])

---
# Part 3 — Day 1 rebuilt in Pandas
Yesterday's `clean()` was ~30 lines. Here is the same pipeline in Pandas, with **the Arjun decision done properly**: an impossible value (105) is treated as **missing**, then imputed, instead of being capped to 100.

In [ ]:
import io
RAW_CSV = """name,roll,math,physics,chemistry,english
Asha,101,91,84,88,79
ravi,102,67,72,,81
Meena,103,84,90,93,88
John,104,45,58,52,70
Priya,105,99,95,97,92
Ravi,102,67,72,,81
Arjun,106,73,64,70,105
Fatima,107,88,79,85,90
"""
SUBJ = ["math", "physics", "chemistry", "english"]

m = pd.read_csv(io.StringIO(RAW_CSV))
m = m.drop_duplicates(subset="roll")                           # 1) de-duplicate
m["name"] = m["name"].str.strip().str.title()                  # 2) normalise names
m[SUBJ] = m[SUBJ].where((m[SUBJ] >= 0) & (m[SUBJ] <= 100))     # 3) impossible -> NaN
m[SUBJ] = m[SUBJ].fillna(m[SUBJ].mean().round(1))              # 4) impute with subject mean
m["average"] = m[SUBJ].mean(axis=1).round(2)
m["grade"] = pd.cut(m["average"], bins=[0, 50, 60, 70, 80, 90, 100],
                    labels=["F", "D", "C", "B", "A", "A+"], right=False)
m.sort_values("average", ascending=False)

In [ ]:
# Toppers per subject, in one line. Compare with yesterday's loop.
print(m.set_index("name")[SUBJ].idxmax())
print("\nGrade counts:\n", m["grade"].value_counts().sort_index())

---
# Part 4 — Matplotlib & Seaborn
Matplotlib draws, and Seaborn adds statistical plots on top with one-line calls. Always use the **figure / axes** style (`fig, ax = plt.subplots()`): it scales to multi-panel figures.

| Question | Plot |
|---|---|
| How is one numeric variable distributed? | histogram / KDE |
| How do categories compare? | bar chart |
| How do two numeric variables relate? | scatter |
| How does something change over time? | line |
| Distribution split by category, with outliers? | box plot |

In [ ]:
rng = np.random.default_rng(0)
x = np.linspace(0, 10, 50)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].hist(rng.normal(70, 10, 500), bins=25, color="steelblue", edgecolor="white")
axes[0].set(title="Histogram: distribution", xlabel="mark", ylabel="count")

axes[1].scatter(x, 2 * x + rng.normal(0, 2, 50), alpha=0.7)
axes[1].plot(x, 2 * x, color="red", label="true line y = 2x")
axes[1].set(title="Scatter + line", xlabel="x", ylabel="y"); axes[1].legend()

m.plot.bar(x="name", y="average", ax=axes[2], legend=False, color="teal")
axes[2].set(title="Bar: average by student", ylabel="average")
plt.tight_layout(); plt.show()

---
# Part 5 — SciPy in one minute
SciPy holds the scientific algorithms: statistics, optimisation, linear algebra and signal processing. Two tools you'll use repeatedly:

In [ ]:
# Statistics: probability under a normal curve (Day 3 goes deeper)
print("P(mark < 85) if marks ~ N(70, 10):", round(stats.norm.cdf(85, loc=70, scale=10), 4))

# Optimisation: the core of ALL machine learning is "find the parameters that minimise an error"
loss = lambda w: (w - 3) ** 2 + 1          # a loss whose minimum is at w = 3
result = optimize.minimize(loss, x0=0.0)
print("minimum found at w =", result.x.round(4), "with loss", round(result.fun, 4))

---
# Part 6 — LIVE DEMO: Titanic Exploratory Data Analysis
**Question:** *Who survived the Titanic, and why?* EDA is the first step of every ML project: you understand the data **before** modelling it. You'll build a survival classifier on this dataset in Week 2.

The EDA checklist we'll follow:
1. Load and inspect (shape, types, sample)
2. Missing values
3. Target variable
4. Univariate: each feature alone
5. Bivariate: each feature vs the target
6. Multivariate: interactions and correlations
7. Statistical tests: are the differences real?
8. Feature engineering and findings

### Step 1 — Load and inspect

In [ ]:
titanic = sns.load_dataset("titanic")
print(titanic.shape)
titanic.head()

In [ ]:
titanic.info()

### Step 2 — Missing values
Always check **how much** is missing and **where**, before deciding what to do.

In [ ]:
missing = titanic.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(titanic) * 100).round(1)
missing[missing["missing"] > 0].sort_values("pct", ascending=False)

Decisions:
- `deck` is **77% missing**. Imputing would invent most of the column, so we drop it.
- `age` is **20% missing** and clearly important, so we impute it, but smartly (Step 8).
- `embarked` / `embark_town` have just 2 missing rows, so we fill them with the most common port.

### Step 3 — The target: how many survived?

In [ ]:
print(titanic["survived"].value_counts())
print(f"\nOverall survival rate: {titanic['survived'].mean():.1%}")

Only **38%** survived, so the classes are imbalanced. A model that always predicts "died" would already score 62% accuracy. Remember this **baseline** in Week 2: a model must beat it to be useful.

### Step 4 — Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
sns.countplot(data=titanic, x="pclass", ax=axes[0]); axes[0].set_title("Passenger class")
sns.countplot(data=titanic, x="sex", ax=axes[1]); axes[1].set_title("Sex")
sns.histplot(data=titanic, x="age", bins=30, kde=True, ax=axes[2]); axes[2].set_title("Age")
sns.histplot(data=titanic, x="fare", bins=40, ax=axes[3]); axes[3].set_title("Fare (right-skewed)")
plt.tight_layout(); plt.show()
print("fare skewness:", round(titanic["fare"].skew(), 2), "-> heavily right-skewed; log transform on Day 10")

### Step 5 — Bivariate: each feature vs survival
The mean of a 0/1 column **is** the survival rate, so `groupby(...).mean()` answers "who survived?" directly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
sns.barplot(data=titanic, x="sex", y="survived", ax=axes[0]); axes[0].set_title("Survival by sex")
sns.barplot(data=titanic, x="pclass", y="survived", ax=axes[1]); axes[1].set_title("Survival by class")
sns.barplot(data=titanic, x="embark_town", y="survived", ax=axes[2]); axes[2].set_title("Survival by port")
plt.tight_layout(); plt.show()

print(titanic.groupby("sex")["survived"].mean().round(3))
print(titanic.groupby("pclass")["survived"].mean().round(3))

In [ ]:
# Age: bin a continuous variable to see the pattern
titanic["age_band"] = pd.cut(titanic["age"], bins=[0, 12, 18, 30, 45, 60, 80],
                             labels=["child", "teen", "18-30", "30-45", "45-60", "60+"])
print(titanic.groupby("age_band", observed=True)["survived"].agg(["mean", "count"]).round(3))

fig, ax = plt.subplots(figsize=(8, 3.6))
sns.kdeplot(data=titanic, x="age", hue="alive", common_norm=False, fill=True, ax=ax)
ax.set_title("Age distribution: survivors vs non-survivors"); plt.show()

### Step 6 — Multivariate: interactions
Is class a separate effect, or does it just reflect sex? A **pivot table** shows both at once.

In [ ]:
pivot = titanic.pivot_table(values="survived", index="sex", columns="class", aggfunc="mean", observed=True)
print(pivot.round(3))

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(pivot, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
ax.set_title("Survival rate by sex x class"); plt.show()

In [ ]:
# Correlation between numeric features (Pearson, from -1 to +1)
num = titanic[["survived", "pclass", "age", "sibsp", "parch", "fare"]]
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix"); plt.show()

Note: correlation only captures **linear** relationships between **numeric** columns. `sex` is the strongest predictor, but it is not even in this matrix because it is text. Encoding categories as numbers is Day 10.

### Step 7 — Statistical tests: is the difference real or chance?

In [ ]:
# Chi-square test of independence: sex vs survival (two categorical variables)
table = pd.crosstab(titanic["sex"], titanic["survived"])
chi2, p, dof, _ = stats.chi2_contingency(table)
print(table, f"\nchi2 = {chi2:.1f}, p-value = {p:.2e}")

# Welch t-test: do survivors' fares differ from non-survivors'? (numeric vs binary)
paid_surv = titanic.loc[titanic.survived == 1, "fare"]
paid_died = titanic.loc[titanic.survived == 0, "fare"]
t, p2 = stats.ttest_ind(paid_surv, paid_died, equal_var=False)
print(f"\nmean fare: survived {paid_surv.mean():.1f} vs died {paid_died.mean():.1f}; t = {t:.2f}, p = {p2:.2e}")

Both p-values are tiny (far below 0.05), so these differences are almost certainly not chance. Day 4 explains exactly what a p-value does and does not mean.

### Step 8 — Cleaning and feature engineering

In [ ]:
clean = titanic.drop(columns=["deck", "age_band", "alive"])   # alive duplicates the target: a LEAK

# Smart age imputation: median age per (class, sex) group, better than one global median
clean["age_missing"] = clean["age"].isna()                     # keep the fact that it was missing
clean["age"] = clean.groupby(["pclass", "sex"])["age"].transform(lambda s: s.fillna(s.median()))

mode_port = clean["embark_town"].mode()[0]
clean["embark_town"] = clean["embark_town"].fillna(mode_port)
clean["embarked"] = clean["embarked"].fillna(clean["embarked"].mode()[0])

# New features
clean["family_size"] = clean["sibsp"] + clean["parch"] + 1
clean["log_fare"] = np.log1p(clean["fare"])

print("missing values left:", clean.isna().sum().sum())
print(clean.groupby("family_size")["survived"].agg(["mean", "count"]).round(2))

In [ ]:
clean.groupby(["pclass", "sex"])["age"].median().round(1)   # the medians used for imputation

### Findings (what you'd present to a manager)
1. **Sex is the strongest signal:** about 74% of women survived vs 19% of men ("women and children first").
2. **Class matters independently:** even among women, 1st class did far better than 3rd (~97% vs 50%).
3. **Children** (under 12) survived at a higher rate than any adult age band.
4. **Family size is non-linear:** small families (2–4) did best; people alone and very large families did worst.
5. **Fare** is linked to survival, but it is largely a proxy for class, so a model must not double-count it.
6. **Data quality:** `deck` was dropped (77% missing); `age` was imputed by class and sex; `alive` was removed as a **target leak**.

**Next:** in Week 2 these become features for k-NN, logistic regression and Random Forest, and you'll see how much accuracy beats the 62% baseline.

---
# Part 7 — Exercises
Paste your code in chat for review.

**Ex 1 — NumPy.** Create a (5, 3) array of random integer marks between 40 and 100 with `seed=7`. Using only NumPy (no loops): print each student's average, the index of the top student, and how many marks are below 50.

**Ex 2 — Pandas.** Using `clean`, find the survival rate for each `who` value (man / woman / child) **within each class**, as a pivot table. Which group had the lowest survival rate?

**Ex 3 — Plot.** Make a single figure with 2 subplots: (a) a box plot of `age` by `pclass`, (b) a bar plot of survival rate by `family_size`. Give every axis a title and labels.

**Ex 4 — Think.** We created `log_fare`. Plot a histogram of `fare` and one of `log_fare` side by side. In one sentence: why might a model prefer `log_fare`?

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

---
### Day 2 recap
- **NumPy:** contiguous typed arrays, vectorised with no loops, typically 10–100x faster than lists. Watch `axis=0` (down the rows) vs `axis=1` (across the columns).
- **Broadcasting** lets you standardise every column in one line.
- **Pandas** is SQL + LINQ in Python: `loc`/masks for WHERE, `groupby` for GROUP BY, `merge` for JOIN.
- **EDA order:** inspect → missing → target → univariate → bivariate → multivariate → tests → features.
- **Leakage:** drop columns that reveal the answer (`alive`). Impossible values become NaN, not a capped number.
- **Tomorrow (Day 3): Probability.** Events, Bayes' theorem, random variables and distributions, with simulations and a hand-built spam filter.